In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# =========================
# --- الخلية 2: Offline Install & Import Fix (NO INTERNET) ---
# =========================
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# ممنوع أي باكج ثقيل/مرتبط GPU أو موجود في بيئة كاجل
FORBIDDEN_PARTS = [
    "torch", "torchvision", "torchaudio",
    "triton", "nvidia_", "cublas", "cudnn", "cuda",
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow"
]

def is_forbidden_wheel(path: str) -> bool:
    fn = os.path.basename(path).lower()
    return any(part in fn for part in FORBIDDEN_PARTS)

def install_wheel_by_part(wheel_name_part: str) -> bool:
    wheels = glob.glob(f"/kaggle/input/**/*{wheel_name_part}*.whl", recursive=True)
    wheels = [w for w in wheels if not is_forbidden_wheel(w)]
    if not wheels:
        print(f"⚠️ No wheel found for: {wheel_name_part}")
        return False
    # نختار أقصر اسم غالباً الحزمة الرئيسية (ليس ملحق إضافي)
    target = sorted(wheels, key=len)[0]
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            target, "--no-deps", "--ignore-installed", "--quiet"
        ])
        reload(site)
        print(f"✅ Installed wheel: {os.path.basename(target)}")
        return True
    except Exception as e:
        print(f"❌ Install failed for {wheel_name_part}: {e}")
        return False

def ensure_import(import_name: str, wheel_part: str) -> bool:
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except ImportError:
        print(f"⚙️ Installing missing: {import_name}")
        ok = install_wheel_by_part(wheel_part)
        if not ok:
            return False
        try:
            __import__(import_name)
            print(f"✅ Imported after install: {import_name}")
            return True
        except Exception as e:
            print(f"❌ Still cannot import {import_name}: {e}")
            return False

# ضروريين للمشروع
ensure_import("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_import("timm", "timm")
ensure_import("ultralytics", "ultralytics")

# استيراد نهائي
import torch
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import segmentation_models_pytorch as smp
from ultralytics import YOLO

print("------ Environment Check ------")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print(f"smp: {smp.__version__}")
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
⚙️ Installing missing: segmentation_models_pytorch
⚠️ No wheel found for: segmentation_models_pytorch
✅ Already available: timm
✅ Already available: ultralytics


ModuleNotFoundError: No module named 'segmentation_models_pytorch'

In [ ]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

In [ ]:
# =========================
# --- الخلية 22: Platinum Engine (Parquet Template + Streaming CSV + LRU Cache) ---
# =========================
import gc, os, csv, glob
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# --- 0) إعدادات ثابتة ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}

print(f"⚡ Device: {DEVICE}")

# --- 1) تحقق صارم: القالب لازم يكون موجود (لأن المسابقة Offline) ---
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError(f"❌ sample_submission.parquet not found at: {SAMPLE_PARQUET_PATH}")

# --- 2) اقرأ قالب الـ Parquet (IDs فقط) ---
print("📦 Reading Parquet template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
n_rows = len(template_ids)
del template
gc.collect()
print(f"✅ Template rows: {n_rows:,}")

# --- 3) اقرأ test.csv لاستخراج fs (لو موجود) ---
fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        id_col = next((c for c in tdf.columns if "id" in c.lower()), None)
        fs_col = next((c for c in tdf.columns if "fs" in c.lower()), None)
        if id_col and fs_col:
            fs_map = dict(zip(tdf[id_col].astype(str), tdf[fs_col].astype(str)))
            print(f"✅ fs_map loaded: {len(fs_map):,} items")
        else:
            print("⚠️ test.csv found but id/fs columns not detected, using default fs=500.")
    except Exception as e:
        print(f"⚠️ Failed reading test.csv: {e} | using default fs=500.")
else:
    print("⚠️ test.csv not found, using default fs=500.")

# --- 4) فهرسة الصور ---
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

# --- 5) تحميل الموديلات (OFFLINE: لا تحميل من الإنترنت) ---
print("⚙️ Loading models (offline)...")

YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

# YOLO
yolo_model = None
if os.path.exists(YOLO_PATH):
    yolo_model = YOLO(YOLO_PATH)
    print("✅ YOLO loaded:", YOLO_PATH)
else:
    # fallback بحث داخل input عن best.pt
    candidates = glob.glob("/kaggle/input/**/*.pt", recursive=True)
    candidates = [p for p in candidates if os.path.basename(p).lower() == "best.pt"]
    if candidates:
        yolo_model = YOLO(candidates[0])
        print("✅ YOLO loaded:", candidates[0])
    else:
        print("⚠️ YOLO model not found. Will use grid fallback crops.")

# UNet
unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse",
)
if os.path.exists(UNET_PATH):
    unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    print("✅ UNet loaded:", UNET_PATH)
else:
    # fallback بحث داخل input عن best_model_phase10.pth
    candidates = glob.glob("/kaggle/input/**/*.pth", recursive=True)
    candidates = [p for p in candidates if os.path.basename(p).lower() == "best_model_phase10.pth"]
    if candidates:
        unet_model.load_state_dict(torch.load(candidates[0], map_location=DEVICE))
        print("✅ UNet loaded:", candidates[0])
    else:
        print("⚠️ UNet weights not found. Predictions may be empty (zeros).")

unet_model.to(DEVICE)
unet_model.eval()

# --- 6) دوال مساعدة ---
def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path_safe(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map:
        return path_map[pid_s]
    try:
        pid_int = str(int(float(pid_s)))
        return path_map.get(pid_int)
    except:
        return None

def apply_high_pass_filter(data, cutoff=0.5, fs=500.0, order=5):
    if len(data) < order * 3:
        return data
    nyq = 0.5 * fs
    if nyq <= 0:
        return data
    normal_cutoff = cutoff / nyq
    if normal_cutoff <= 0 or normal_cutoff >= 1:
        return data
    b, a = butter(order, normal_cutoff, btype="high", analog=False)
    return filtfilt(b, a, data)

def smart_einthoven_fix(leads_dict):
    # (I + III) ≈ II
    if 'I' in leads_dict and 'II' in leads_dict and 'III' in leads_dict:
        l = min(len(leads_dict['I']), len(leads_dict['II']), len(leads_dict['III']))
        I = leads_dict['I'][:l]
        II = leads_dict['II'][:l]
        III = leads_dict['III'][:l]
        residual = (I + III) - II
        leads_dict['II'][:l] += residual / 3.0
        leads_dict['I'][:l] -= residual / 3.0
        leads_dict['III'][:l] -= residual / 3.0
    return leads_dict

def advanced_perspective_correction(image):
    if image is None:
        return None
    h, w = image.shape[:2]
    if h < 10 or w < 10:
        return image
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (max(32, w//4), max(32, h//4)))
    edges = cv2.Canny(small, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50,
                            minLineLength=max(10, small.shape[1]//8), maxLineGap=10)
    if lines is None:
        return image
    angles = []
    for l in lines:
        x1, y1, x2, y2 = l[0]
        ang = np.degrees(np.arctan2((y2-y1), (x2-x1)))
        if abs(ang) < 15:
            angles.append(ang)
    if len(angles) < 5:
        return image
    median_ang = float(np.median(angles))
    if abs(median_ang) < 0.5:
        return image
    M = cv2.getRotationMatrix2D((w//2, h//2), median_ang, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def robust_multi_point_calibration(crop, default_val=25.0):
    if crop is None or crop.size == 0:
        return default_val
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    # تعزيز بسيط
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sx = np.sum(np.abs(edges_x), axis=0)
    sy = np.sum(np.abs(edges_y), axis=1)
    peaks_x, _ = find_peaks(sx, distance=10, prominence=50)
    peaks_y, _ = find_peaks(sy, distance=10, prominence=50)
    grid_sizes = []
    if len(peaks_x) > 3:
        dx = np.diff(peaks_x)
        grid_sizes.extend(dx[(dx > 10) & (dx < 100)])
    if len(peaks_y) > 3:
        dy = np.diff(peaks_y)
        grid_sizes.extend(dy[(dy > 10) & (dy < 100)])
    if len(grid_sizes) >= 5:
        return float(np.median(np.array(grid_sizes)))
    return default_val

def preprocess_remove_grid_rgb(image_rgb):
    if image_rgb is None:
        return None
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    # أحمر الشبكة غالباً
    mask = (
        cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
        cv2.inRange(hsv, (170, 50, 50), (180, 255, 255))
    )
    out = image_rgb.copy()
    out[mask > 0] = (255, 255, 255)
    return out

def get_yolo_crops_with_fallback(image, model):
    h, w = image.shape[:2]
    crops = [None] * 12
    if model is not None:
        try:
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640, device=DEVICE)
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes.data.detach().cpu().numpy()
                # boxes: [x1,y1,x2,y2,conf,cls]
                best = {}
                for b in boxes:
                    cls_id = int(b[5])
                    conf = float(b[4])
                    if 0 <= cls_id < 12:
                        if cls_id not in best or conf > best[cls_id][0]:
                            best[cls_id] = (conf, b[:4])
                for cls_id, (_, xyxy) in best.items():
                    x1,y1,x2,y2 = map(int, xyxy)
                    pad = 5
                    x1 = max(0, x1-pad); y1 = max(0, y1-pad)
                    x2 = min(w, x2+pad); y2 = min(h, y2+pad)
                    crops[cls_id] = image[y1:y2, x1:x2]
                if sum(c is not None for c in crops) >= 10:
                    return crops
        except:
            pass
    # fallback grid split 4x3
    rh = h // 4
    cw = w // 3
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def batch_predict_unet(crops, model):
    if not crops:
        return [], []
    target_h = 256
    processed = []
    scales = []
    shapes = []
    for img in crops:
        h, w = img.shape[:2]
        if h <= 0 or w <= 0:
            continue
        scale = target_h / h
        new_w = int(max(1, w * scale))
        img_rz = cv2.resize(img, (new_w, target_h))
        tens = torch.from_numpy(img_rz).permute(2,0,1).float() / 255.0
        processed.append(tens)
        scales.append(scale)
        shapes.append((target_h, new_w))

    if not processed:
        return [], []

    max_w = max(s[1] for s in shapes)
    max_w = int(np.ceil(max_w / 32) * 32)

    batch = torch.zeros((len(processed), 3, target_h, max_w), dtype=torch.float32, device=DEVICE)
    for i, t in enumerate(processed):
        batch[i, :, :t.shape[1], :t.shape[2]] = t.to(DEVICE)

    with torch.no_grad():
        p1 = torch.sigmoid(model(batch))
        p2 = torch.sigmoid(model(torch.flip(batch, [3])))
        p = (p1 + torch.flip(p2, [3])) / 2.0

    p = p.detach().cpu().numpy()
    prob_maps = [p[i, 0, :shapes[i][0], :shapes[i][1]] for i in range(len(processed))]
    return prob_maps, scales

def fast_viterbi(prob_map):
    # نسخة DP سريعة (بدون تحميل كبير)
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6)
    dp = np.zeros_like(cost, dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int32)
    dp[:, 0] = cost[:, 0]
    smooth = 0.5
    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1); c_m1[0] = np.inf
        c_0  = prev
        c_p1 = np.roll(prev, -1); c_p1[-1] = np.inf
        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0)
        dp[:, t] = cost[:, t] + stacked[which, np.arange(H)]
        parent[:, t] = np.clip(np.arange(H) + (which - 1), 0, H-1)
    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = parent[path[t+1], t+1]
    # convert to y (bottom-up)
    return (H - path).astype(np.float32)

# --- 7) LRU Cache لمعالجة كل مريض مرة واحدة فقط ---
patient_cache = OrderedDict()
MAX_CACHE_PATIENTS = 16

def compute_patient_leads(pid_clean_str):
    default_leads = np.zeros((12, 5000), dtype=np.float32)
    path = get_image_path_safe(pid_clean_str)
    if not path:
        return default_leads

    # fs
    fs_val = 500.0
    try:
        fs_val = float(fs_map.get(pid_clean_str, 500.0))
    except:
        fs_val = 500.0

    try:
        img = cv2.imread(path)
        if img is None:
            return default_leads
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = advanced_perspective_correction(img)
        global_grid = robust_multi_point_calibration(img, 25.0)

        crops = get_yolo_crops_with_fallback(img, yolo_model)

        clean_crops = []
        valid_idx = []
        for i, crop in enumerate(crops):
            if crop is not None and crop.size > 100:
                clean_crops.append(preprocess_remove_grid_rgb(crop))
                valid_idx.append(i)

        if not clean_crops:
            return default_leads

        prob_maps, scales = batch_predict_unet(clean_crops, unet_model)
        if not prob_maps:
            return default_leads

        leads_dict = {}
        for j, real_idx in enumerate(valid_idx):
            prob = prob_maps[j]
            scale = scales[j]
            lname = LEAD_NAMES[real_idx]

            local_grid = robust_multi_point_calibration(clean_crops[j], default_val=global_grid)
            raw = fast_viterbi(prob)

            g_sc = local_grid * scale
            divider = (g_sc * 10.0) if g_sc > 1.0 else 250.0
            sig_mv = (raw - np.median(raw)) / float(divider)
            sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)

            if len(sig_mv) > 15:
                sig_mv = savgol_filter(sig_mv, 11, 3)
            if len(sig_mv) > 20:
                sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=fs_val)

            if len(sig_mv) > 0:
                sig_rs = resample(sig_mv, 5000).astype(np.float32)
                sig_rs = np.nan_to_num(sig_rs, nan=0.0, posinf=0.0, neginf=0.0)
                leads_dict[lname] = sig_rs

        leads_dict = smart_einthoven_fix(leads_dict)

        final = np.zeros((12, 5000), dtype=np.float32)
        for i, l in enumerate(LEAD_NAMES):
            if l in leads_dict:
                final[i] = leads_dict[l]
        final = np.nan_to_num(final, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        return final

    except:
        return default_leads

# --- 8) كتابة submission.csv (Streaming) ---
print("🚀 Writing submission.csv (streaming)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        # لا تغيّر الـ id أبداً
        row_id = str(rid)

        try:
            parts = row_id.rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([row_id, "0.0000"])
                continue

            pid_part = parts[0]
            idx_part = int(parts[1])
            lead_part = parts[2]

            pid_clean_str = clean_pid(pid_part)

            # cache
            if pid_clean_str in patient_cache:
                leads_matrix = patient_cache[pid_clean_str]
                patient_cache.move_to_end(pid_clean_str)
            else:
                leads_matrix = compute_patient_leads(pid_clean_str)
                patient_cache[pid_clean_str] = leads_matrix
                if len(patient_cache) > MAX_CACHE_PATIENTS:
                    patient_cache.popitem(last=False)

            val = 0.0
            if 0 <= idx_part < 5000:
                lead_idx = LEAD_TO_IDX.get(lead_part, 0)
                v = float(leads_matrix[lead_idx][idx_part])
                if np.isfinite(v):
                    val = v

            writer.writerow([row_id, f"{val:.4f}"])

        except:
            writer.writerow([row_id, "0.0000"])

# تنظيف
del patient_cache
gc.collect()
torch.cuda.empty_cache()

print("✅ Done. submission.csv created successfully.")


In [ ]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")
